<a href="https://colab.research.google.com/github/RaphaelRAY/projeto-bi-games/blob/main/notebooks/1_importar_dados_bq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Importação de Dados do Kaggle para o BigQuery
Dataset: Mercado de Games

Este notebook utiliza a biblioteca `kagglehub` para baixar os arquivos originais (.csv) do Kaggle em tempo real e em seguida enviá-los diretamente para o Google BigQuery.

## 1. Importação das Bibliotecas

In [2]:
import pandas as pd
import os
import kagglehub
from google.cloud import bigquery

# Se você estiver usando o Google Colab, descomente as linhas abaixo para autenticar sua conta GCP
from google.colab import auth
auth.authenticate_user()

## 2. Autenticação no BigQuery

In [3]:
# Substitua pelas suas credenciais reais do Google Cloud Platform
project_id = 'directed-mender-489100-m3'
dataset_id = 'games_data'

client = bigquery.Client(project=project_id)

## 3. Baixar os Dados Diretamente do Kaggle

In [4]:
print("Iniciando o download da base de dados mais recente do Kaggle...")

# Faz o download automático e retorna o caminho onde os arquivos foram salvos na máquina/ambiente
data_path = kagglehub.dataset_download("gabrigoncam/video-game-dataset-multidimensional")

print("Arquivos baixados na pasta local:", data_path)

# Extrair o caminho de todos os CSVs fornecidos no dataset
all_files = []
for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith('.csv'):
            all_files.append(os.path.join(root, file))

print(f"\nEncontrados {len(all_files)} arquivos .csv prontos para envio ao BigQuery.")

Iniciando o download da base de dados mais recente do Kaggle...


100%|██████████| 210M/210M [00:01<00:00, 141MB/s]

Extracting files...


Arquivos baixados na pasta local: /root/.cache/kagglehub/datasets/gabrigoncam/video-game-dataset-multidimensional/versions/1

Encontrados 11 arquivos .csv prontos para envio ao BigQuery.


## 4. Subir Tudo para o BigQuery

In [8]:
for file_path in all_files:
    # Extrair o nome do arquivo para usar como nome da tabela no BigQuery
    file_name = os.path.basename(file_path)
    table_name = file_name.replace('.csv', '')

    table_id = f"{project_id}.{dataset_id}.{table_name}"

    print(f"Enviando [{file_name}] para a tabela [{table_id}]...")

    # Handle ParserError for games.csv by using the python engine and skipping bad lines
    if file_name == 'games.csv':
        df = pd.read_csv(file_path, engine='python', on_bad_lines='skip')
    elif file_name == 'game_platforms.csv':
        # Handle ArrowTypeError for game_platforms.csv by specifying dtype for 'game_id'
        df = pd.read_csv(file_path, dtype={'game_id': str})
    elif file_name == 'game_developers.csv':
        # Handle ParserError for game_developers.csv by using the python engine and skipping bad lines
        df = pd.read_csv(file_path, engine='python', on_bad_lines='skip')
    else:
        df = pd.read_csv(file_path)

    # Substituir (truncate) a tabela caso ela já exista no dataset
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")

    # Dispara o job pro BQ
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()

    print(f"\t>> ✅ Sucesso! Tabela {table_name} carregada.\n")

print("Todos os dados foram enviados para o Google BigQuery com sucesso!")

Enviando [game_stores.csv] para a tabela [directed-mender-489100-m3.games_data.game_stores]...
	>> ✅ Sucesso! Tabela game_stores carregada.

Enviando [game_genres.csv] para a tabela [directed-mender-489100-m3.games_data.game_genres]...
	>> ✅ Sucesso! Tabela game_genres carregada.

Enviando [game_derived_metrics.csv] para a tabela [directed-mender-489100-m3.games_data.game_derived_metrics]...
	>> ✅ Sucesso! Tabela game_derived_metrics carregada.

Enviando [game_tags.csv] para a tabela [directed-mender-489100-m3.games_data.game_tags]...
	>> ✅ Sucesso! Tabela game_tags carregada.

Enviando [games.csv] para a tabela [directed-mender-489100-m3.games_data.games]...
	>> ✅ Sucesso! Tabela games carregada.

Enviando [game_platforms.csv] para a tabela [directed-mender-489100-m3.games_data.game_platforms]...
	>> ✅ Sucesso! Tabela game_platforms carregada.

Enviando [game_developers.csv] para a tabela [directed-mender-489100-m3.games_data.game_developers]...
	>> ✅ Sucesso! Tabela game_developers c